In [ ]:
import os
import json
from pathlib import Path


import numpy as np
import pandas as pd


from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer


from sklearn.metrics import (
accuracy_score, precision_score, recall_score, f1_score,
roc_auc_score, classification_report, confusion_matrix
)


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


import joblib

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.20


ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

Load

In [ ]:
DATA_PATH = Path("data/dataset.csv")

df = pd.read_csv(DATA_PATH)
df.head()

EDA básico...

In [ ]:
print("Shape:", df.shape)
df.info()
df.describe(include="all").T.head(20)

na_rate = (df.isna().mean().sort_values(ascending=False))
na_rate.head(15)

In [ ]:
TARGET_COL = "target"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Identificar tipos
num_features = X.select_dtypes(include=["number"]).columns.tolist()
cat_features = X.select_dtypes(exclude=["number"]).columns.tolist()

num_features, cat_features

In [ ]:
# por defecto 75-25
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
stratify=y
)

X_train.shape, X_test.shape

In [ ]:
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()) ])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")) ])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_features),
        ("cat", categorical_pipe, cat_features),    ],
    remainder="drop" )

In [ ]:
# Modelo 1: Baseline (Logistic Regression)
baseline = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)) ])

baseline.fit(X_train, y_train)

pred = baseline.predict(X_test)
proba = baseline.predict_proba(X_test)[:, 1]

metrics_baseline = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "precision": float(precision_score(y_test, pred, zero_division=0)),
    "recall": float(recall_score(y_test, pred, zero_division=0)),
    "f1": float(f1_score(y_test, pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, proba)), }

metrics_baseline

In [ ]:
print(classification_report(y_test, pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))

In [ ]:
# Modelo 2: Fuerte (Random Forest)
rf = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
    n_estimators=400,
    random_state=RANDOM_STATE,
    n_jobs=-1))
])

rf.fit(X_train, y_train)

pred = rf.predict(X_test)
proba = rf.predict_proba(X_test)[:, 1]

metrics_rf = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "precision": float(precision_score(y_test, pred, zero_division=0)),
    "recall": float(recall_score(y_test, pred, zero_division=0)),
    "f1": float(f1_score(y_test, pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, proba)),
}

metrics_rf

In [ ]:
# Validación cruzada (detección rápida de overfitting)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scores = cross_val_score(baseline, X_train, y_train, cv=cv, scoring="f1")
print("Baseline F1 CV mean/std:", scores.mean(), scores.std())

scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring="f1")
print("RF F1 CV mean/std:", scores.mean(), scores.std())

In [ ]:
# Comparativa de resultados
compara = pd.DataFrame([
    {"model": "LogisticRegression", **metrics_baseline},
    {"model": "RandomForest", **metrics_rf},
]).sort_values(by="f1", ascending=False)

compara

In [ ]:
# guarda
best_model = rf if metrics_rf["f1"] >= metrics_baseline["f1"] else baseline
joblib.dump(best_model, ARTIFACT_DIR / "model.joblib")

with open(ARTIFACT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "baseline": metrics_baseline,
        "random_forest": metrics_rf,
        "selected": "RandomForest" if best_model is rf else "LogisticRegression"
    }, f, indent=2)

pred_test = best_model.predict(X_test)
proba_test = best_model.predict_proba(X_test)[:, 1]
out = X_test.copy()
out["y_true"] = y_test.values
out["y_pred"] = pred_test
out["y_proba"] = proba_test
out.to_csv(ARTIFACT_DIR / "test_predictions.csv", index=False)

### Extra

In [ ]:
# GridSearchCV

from sklearn.model_selection import GridSearchCV

param_grid = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [None, 8, 12],
    "clf__min_samples_leaf": [1, 5, 10], }

grid = GridSearchCV(
    estimator=rf,              # pipeline completo
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1 )

grid.fit(X_train, y_train)
grid.best_params_

best_model = grid.best_estimator_

pred = best_model.predict(X_test)
proba = best_model.predict_proba(X_test)[:, 1]

print("F1:", f1_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, proba))

In [ ]:
# Data Validation

import pandera as pa
from pandera import Column, DataFrameSchema

schema = DataFrameSchema({
    "f0": Column(float, nullable=False),
    "f1": Column(float),
    "region": Column(str, nullable=False),
    "target": Column(int, checks=pa.Check.isin([0, 1]))
})

schema.validate(df)
